# Parkinson's Data Visualization Notebook

This notebook is built for report-ready exploratory analysis of the PPMI data.

It focuses on:
- data overview and quality
- demographics and cohort composition
- correlation structure between clinical variables
- disease progression over time (motor and cognitive)
- participant-level progression rates

## 1. Setup

Run all cells from top to bottom. The notebook expects CSV files in `data/PPMI_data/`.

In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 12

DATA_DIR = Path("../data/PPMI_data")
if not DATA_DIR.exists():
    DATA_DIR = Path("data/PPMI_data")

assert DATA_DIR.exists(), f"Could not find data directory. Tried: {Path('../data/PPMI_data').resolve()} and {Path('data/PPMI_data').resolve()}"
print("Using data folder:", DATA_DIR.resolve())

In [ ]:
files = {
    "demographics": "Demographics_12Mar2026.csv",
    "age_visit": "Age_at_visit_12Mar2026.csv",
    "status": "Participant_Status_12Mar2026.csv",
    "updrs3": "MDS-UPDRS_Part_III_12Mar2026.csv",
    "moca": "Montreal_Cognitive_Assessment__MoCA__12Mar2026.csv",
    "ledd": "LEDD_Concomitant_Medication_Log_12Mar2026.csv",
    "datscan": "Xing_Core_Lab_-_Quant_SBR_12Mar2026.csv",
}

dfs = {}
for name, filename in files.items():
    path = DATA_DIR / filename
    dfs[name] = pd.read_csv(path)

overview = pd.DataFrame({
    "table": list(dfs.keys()),
    "rows": [df.shape[0] for df in dfs.values()],
    "columns": [df.shape[1] for df in dfs.values()],
})
overview

## 2. Data Overview and Missingness

This section summarizes table sizes and how complete key analysis columns are.

In [ ]:
key_columns = {
    "demographics": ["PATNO", "EVENT_ID", "SEX"],
    "age_visit": ["PATNO", "EVENT_ID", "AGE_AT_VISIT"],
    "status": ["PATNO", "COHORT", "ENROLL_STATUS"],
    "updrs3": ["PATNO", "EVENT_ID", "NP3TOT"],
    "moca": ["PATNO", "EVENT_ID", "MCATOT"],
    "ledd": ["PATNO", "EVENT_ID", "LEDD"],
    "datscan": ["PATNO", "EVENT_ID", "PUTAMEN_L_REF_CWM", "PUTAMEN_R_REF_CWM"],
}

rows = []
for table, cols in key_columns.items():
    df = dfs[table]
    for c in cols:
        if c in df.columns:
            missing_pct = df[c].isna().mean() * 100
            rows.append({"table": table, "column": c, "missing_percent": missing_pct})

missing_df = pd.DataFrame(rows).sort_values(["table", "missing_percent"], ascending=[True, False])
display(missing_df.head(20))

plt.figure(figsize=(12, 6))
plot_df = missing_df[missing_df["column"].isin(["NP3TOT", "MCATOT", "AGE_AT_VISIT", "LEDD", "PUTAMEN_L_REF_CWM", "PUTAMEN_R_REF_CWM"])]
sns.barplot(data=plot_df, x="column", y="missing_percent", hue="table")
plt.title("Missingness in Key Analysis Variables")
plt.ylabel("Missing (%)")
plt.xlabel("")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 3. Demographics and Cohort Composition

Demographic profile is based on participant-level tables:
- `Participant_Status` for cohort assignment
- `Demographics` for sex
- baseline/first available age from `Age_at_visit`

In [ ]:
status = dfs["status"].copy()
demo = dfs["demographics"].copy()
age = dfs["age_visit"].copy()

# Participant-level cohort (1 row per participant)
status_pt = status.drop_duplicates("PATNO")[["PATNO", "COHORT", "ENROLL_STATUS"]]

# Sex mapping (PPMI coding is typically 0/1)
sex_map = {0: "Female", 1: "Male", "0": "Female", "1": "Male"}
demo["SEX_LABEL"] = demo["SEX"].map(sex_map).fillna("Unknown")
sex_pt = demo.drop_duplicates("PATNO")[["PATNO", "SEX_LABEL"]]

# First available age per participant
age = age.copy()
age["AGE_AT_VISIT"] = pd.to_numeric(age["AGE_AT_VISIT"], errors="coerce")
age_pt = age.sort_values(["PATNO", "AGE_AT_VISIT"]).drop_duplicates("PATNO")[["PATNO", "AGE_AT_VISIT"]]

participants = status_pt.merge(sex_pt, on="PATNO", how="left").merge(age_pt, on="PATNO", how="left")

print(f"Unique participants: {participants['PATNO'].nunique():,}")
print(f"Age available: {participants['AGE_AT_VISIT'].notna().mean()*100:.1f}%")
participants.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5))

cohort_counts = participants["COHORT"].fillna("Unknown").astype(str).value_counts().reset_index()
cohort_counts.columns = ["COHORT", "COUNT"]
sns.barplot(data=cohort_counts, x="COHORT", y="COUNT", ax=axes[0], color="#4C78A8")
axes[0].set_title("Cohort Composition")
axes[0].set_xlabel("COHORT")
axes[0].set_ylabel("Participants")
axes[0].tick_params(axis="x", rotation=45)

sex_counts = participants["SEX_LABEL"].fillna("Unknown").value_counts().reset_index()
sex_counts.columns = ["SEX_LABEL", "COUNT"]
sns.barplot(data=sex_counts, x="SEX_LABEL", y="COUNT", ax=axes[1], palette="Set2")
axes[1].set_title("Sex Distribution")
axes[1].set_xlabel("")
axes[1].set_ylabel("Participants")

sns.histplot(participants["AGE_AT_VISIT"].dropna(), bins=30, kde=True, ax=axes[2], color="#F58518")
axes[2].set_title("Age Distribution (First Available Visit)")
axes[2].set_xlabel("Age")
axes[2].set_ylabel("Count")

plt.tight_layout()
plt.show()

## 4. Disease Progression Over Time

Motor progression uses MDS-UPDRS Part III total score (`NP3TOT`).

To visualize visits chronologically, `EVENT_ID` is converted to an approximate numeric visit order:
- `SC` -> -1
- `BL` -> 0
- `Vxx` -> xx
- other visit labels are left as missing for ordered trend plots

In [ ]:
updrs = dfs["updrs3"].copy()
updrs["NP3TOT"] = pd.to_numeric(updrs["NP3TOT"], errors="coerce")

cohort_map = status.drop_duplicates("PATNO")[["PATNO", "COHORT"]]
updrs = updrs.merge(cohort_map, on="PATNO", how="left")

def visit_to_order(event_id):
    if pd.isna(event_id):
        return np.nan
    event = str(event_id).strip().upper()
    if event == "SC":
        return -1
    if event == "BL":
        return 0
    m = re.match(r"V(\d+)$", event)
    if m:
        return int(m.group(1))
    return np.nan

updrs["VISIT_ORDER"] = updrs["EVENT_ID"].apply(visit_to_order)
updrs = updrs.dropna(subset=["NP3TOT", "VISIT_ORDER"]).copy()

major_cohorts = updrs["COHORT"].astype(str).value_counts()
major_cohorts = major_cohorts[major_cohorts >= 100].index
plot_updrs = updrs[updrs["COHORT"].astype(str).isin(major_cohorts)]

plt.figure(figsize=(12, 6))
sns.lineplot(
    data=plot_updrs,
    x="VISIT_ORDER",
    y="NP3TOT",
    hue="COHORT",
    estimator="mean",
    errorbar=("ci", 95),
)
plt.title("Mean UPDRS III Progression by Cohort")
plt.xlabel("Visit Order")
plt.ylabel("UPDRS III (NP3TOT)")
plt.legend(title="COHORT", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Individual trajectories for a sample of participants
sample_ids = (
    updrs.groupby("PATNO")["VISIT_ORDER"].nunique()
    .sort_values(ascending=False)
    .head(30)
    .index
)

sample = updrs[updrs["PATNO"].isin(sample_ids)].sort_values(["PATNO", "VISIT_ORDER"])

plt.figure(figsize=(12, 7))
for pid, sub in sample.groupby("PATNO"):
    plt.plot(sub["VISIT_ORDER"], sub["NP3TOT"], alpha=0.45)

plt.title("Example Individual UPDRS III Trajectories (Top 30 by visit count)")
plt.xlabel("Visit Order")
plt.ylabel("UPDRS III (NP3TOT)")
plt.tight_layout()
plt.show()

## 5. Cognition and Clinical Associations

Cognitive progression uses MoCA total (`MCATOT`).
We also inspect the relationship between motor severity (`NP3TOT`) and cognition (`MCATOT`).

In [ ]:
moca = dfs["moca"].copy()
moca["MCATOT"] = pd.to_numeric(moca["MCATOT"], errors="coerce")
moca["VISIT_ORDER"] = moca["EVENT_ID"].apply(visit_to_order)
moca = moca.dropna(subset=["MCATOT", "VISIT_ORDER"]).copy()

plt.figure(figsize=(12, 6))
sns.lineplot(data=moca, x="VISIT_ORDER", y="MCATOT", estimator="mean", errorbar=("ci", 95), color="#54A24B")
plt.title("Mean MoCA Score Across Visits")
plt.xlabel("Visit Order")
plt.ylabel("MoCA Total (MCATOT)")
plt.tight_layout()
plt.show()

motor_cog = updrs[["PATNO", "EVENT_ID", "NP3TOT"]].merge(
    moca[["PATNO", "EVENT_ID", "MCATOT"]], on=["PATNO", "EVENT_ID"], how="inner"
)

plt.figure(figsize=(8, 6))
sns.regplot(data=motor_cog, x="MCATOT", y="NP3TOT", scatter_kws={"alpha": 0.35}, line_kws={"color": "black"})
plt.title("Motor Severity vs Cognition")
plt.xlabel("MoCA Total (higher = better cognition)")
plt.ylabel("UPDRS III (higher = worse motor score)")
plt.tight_layout()
plt.show()

print(f"Paired UPDRS/MoCA observations: {len(motor_cog):,}")
print(f"Pearson correlation (MCATOT vs NP3TOT): {motor_cog[['MCATOT','NP3TOT']].corr().iloc[0,1]:.3f}")

## 6. Multi-Modal Correlation (Clinical + Imaging + Medication)

We create a visit-level merged table and compute correlations across key numeric features:
- age (`AGE_AT_VISIT`)
- motor severity (`NP3TOT`)
- cognition (`MCATOT`)
- medication burden (`LEDD`)
- DaTscan putamen binding (left/right)

In [ ]:
age2 = dfs["age_visit"][["PATNO", "EVENT_ID", "AGE_AT_VISIT"]].copy()
age2["AGE_AT_VISIT"] = pd.to_numeric(age2["AGE_AT_VISIT"], errors="coerce")

ledd = dfs["ledd"][["PATNO", "EVENT_ID", "LEDD"]].copy()
ledd["LEDD"] = pd.to_numeric(ledd["LEDD"], errors="coerce")
ledd = ledd.groupby(["PATNO", "EVENT_ID"], as_index=False)["LEDD"].sum()

datscan_cols = ["PATNO", "EVENT_ID", "PUTAMEN_L_REF_CWM", "PUTAMEN_R_REF_CWM", "CAUDATE_L_REF_CWM", "CAUDATE_R_REF_CWM"]
datscan = dfs["datscan"][datscan_cols].copy()
for c in datscan_cols[2:]:
    datscan[c] = pd.to_numeric(datscan[c], errors="coerce")

datscan["PUTAMEN_MEAN"] = datscan[["PUTAMEN_L_REF_CWM", "PUTAMEN_R_REF_CWM"]].mean(axis=1)
datscan["CAUDATE_MEAN"] = datscan[["CAUDATE_L_REF_CWM", "CAUDATE_R_REF_CWM"]].mean(axis=1)

base = updrs[["PATNO", "EVENT_ID", "NP3TOT"]].copy()
merged = (
    base.merge(moca[["PATNO", "EVENT_ID", "MCATOT"]], on=["PATNO", "EVENT_ID"], how="left")
        .merge(age2, on=["PATNO", "EVENT_ID"], how="left")
        .merge(ledd, on=["PATNO", "EVENT_ID"], how="left")
        .merge(datscan[["PATNO", "EVENT_ID", "PUTAMEN_MEAN", "CAUDATE_MEAN"]], on=["PATNO", "EVENT_ID"], how="left")
)

corr_cols = ["NP3TOT", "MCATOT", "AGE_AT_VISIT", "LEDD", "PUTAMEN_MEAN", "CAUDATE_MEAN"]
corr_mat = merged[corr_cols].corr(method="pearson")

plt.figure(figsize=(8, 6))
sns.heatmap(corr_mat, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix of Key Clinical Variables")
plt.tight_layout()
plt.show()

merged[corr_cols].describe().T

## 7. Participant-Level Progression Rate

For each participant with at least 3 visits, we fit a simple linear slope:
`NP3TOT ~ VISIT_ORDER`.

Interpretation:
- positive slope -> worsening motor score over time
- negative slope -> improving motor score over time (or treatment effect / variability)

In [ ]:
prog = updrs[["PATNO", "VISIT_ORDER", "NP3TOT", "COHORT"]].dropna().copy()

slopes = []
for pid, sub in prog.groupby("PATNO"):
    sub = sub.sort_values("VISIT_ORDER")
    if sub["VISIT_ORDER"].nunique() < 3:
        continue
    x = sub["VISIT_ORDER"].to_numpy()
    y = sub["NP3TOT"].to_numpy()
    slope = np.polyfit(x, y, 1)[0]
    cohort = sub["COHORT"].iloc[0]
    slopes.append({"PATNO": pid, "UPDRS_SLOPE_PER_VISIT": slope, "COHORT": cohort, "N_VISITS": len(sub)})

slopes_df = pd.DataFrame(slopes)
print(f"Participants with >=3 visits and slope estimate: {len(slopes_df):,}")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(slopes_df["UPDRS_SLOPE_PER_VISIT"], bins=40, kde=True, ax=axes[0], color="#E45756")
axes[0].set_title("Distribution of UPDRS Progression Slopes")
axes[0].set_xlabel("Slope per visit")

cohort_counts = slopes_df["COHORT"].astype(str).value_counts()
keep = cohort_counts[cohort_counts >= 40].index
box_df = slopes_df[slopes_df["COHORT"].astype(str).isin(keep)].copy()

sns.boxplot(data=box_df, x="COHORT", y="UPDRS_SLOPE_PER_VISIT", ax=axes[1], color="#72B7B2")
axes[1].set_title("Progression Slopes by Cohort")
axes[1].set_xlabel("COHORT")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

slopes_df.describe(include="all")

## 8. Report-Friendly Summary Points

Use the values printed below directly in your report text.

In [ ]:
n_participants = participants["PATNO"].nunique()
cohort_top = participants["COHORT"].astype(str).value_counts().head(3)
median_age = participants["AGE_AT_VISIT"].median()
sex_dist = participants["SEX_LABEL"].value_counts(normalize=True) * 100

motor_cog_corr = motor_cog[["MCATOT", "NP3TOT"]].corr().iloc[0, 1]
median_slope = slopes_df["UPDRS_SLOPE_PER_VISIT"].median()

print(f"Total participants: {n_participants:,}")
print("Top 3 cohorts by size:")
print(cohort_top.to_string())
print(f"Median age (first available): {median_age:.1f} years")
print("Sex distribution (%):")
print(sex_dist.round(1).to_string())
print(f"Correlation MCATOT vs NP3TOT: {motor_cog_corr:.3f}")
print(f"Median UPDRS slope per visit: {median_slope:.3f}")